# 02 - Preprocessing And Feature Engineering

This notebook prepares modeling data from `data/processed/combined_all_cities.csv` with explicit leakage controls:
- drop pre-existing `Month` column and derive time features from `Timestamp`
- split by date before fitting preprocessing statistics
- fit outlier capping bounds and scaler on training data only
- generate lag/rolling features using strictly past observations within each city


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

INPUT_PATH = Path('data/processed/combined_all_cities.csv')
SPLITS_DIR = Path('data/splits')
MODELS_DIR = Path('outputs/models')

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp('2020-01-01')
TRAIN_END = pd.Timestamp('2023-12-31')
TEST_START = pd.Timestamp('2024-01-01')
TEST_END = pd.Timestamp('2024-12-31')

REQUIRED_COLUMNS = ['Timestamp', 'City', 'Location', 'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3', 'AQI']
POLLUTANT_COLUMNS = ['PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3']
LAG_ROLL_COLS = [
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30',
    'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean'
]
FEATURE_COLUMNS = [
    'PM2.5', 'PM10', 'NO2', 'NH3', 'SO2', 'CO', 'O3',
    'PM2.5_lag1', 'PM2.5_lag7', 'PM2.5_lag30',
    'AQI_lag1', 'AQI_lag7',
    'PM2.5_roll7_mean', 'PM2.5_roll7_std', 'PM2.5_roll30_mean',
    'day_of_week', 'month', 'season', 'is_weekend', 'year'
]


## Load, Validate, And Normalize Schema

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing input dataset: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)

if 'Month' in df.columns:
    df = df.drop(columns=['Month'])
    print('Dropped existing Month column to avoid duplicated temporal features.')

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
bad_ts = int(df['Timestamp'].isna().sum())
if bad_ts:
    print(f'Dropping rows with invalid Timestamp: {bad_ts}')
df = df.dropna(subset=['Timestamp']).copy()

for col in POLLUTANT_COLUMNS + ['AQI']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values(['City', 'Timestamp']).reset_index(drop=True)
print('Shape after load/parse:', df.shape)
print('Date range:', df['Timestamp'].min().date(), 'to', df['Timestamp'].max().date())


## Split First (Leakage Control)

Time-aware split is done before fitting preprocessing statistics. This avoids contaminating training transforms with future-period distribution from 2024 test data.

In [ ]:
train_raw = df[(df['Timestamp'] >= TRAIN_START) & (df['Timestamp'] <= TRAIN_END)].copy()
test_raw = df[(df['Timestamp'] >= TEST_START) & (df['Timestamp'] <= TEST_END)].copy()

assert not train_raw.empty, 'Training split is empty.'
assert not test_raw.empty, 'Test split is empty.'

assert train_raw['Timestamp'].max() < test_raw['Timestamp'].min(), 'Train/Test date overlap detected.'
assert train_raw['Timestamp'].max() <= TRAIN_END, 'Training contains dates after split end.'
assert test_raw['Timestamp'].min() >= TEST_START, 'Test contains dates before split start.'

print(f'Train raw rows: {len(train_raw):,} | {train_raw["Timestamp"].min().date()} to {train_raw["Timestamp"].max().date()}')
print(f'Test raw rows:  {len(test_raw):,} | {test_raw["Timestamp"].min().date()} to {test_raw["Timestamp"].max().date()}')


## Missing Value Handling (Per Split, Per City)

In [ ]:
def fill_missing_per_split(split_df, cols):
    out = split_df.sort_values(['City', 'Timestamp']).copy()
    before = out[cols + ['AQI']].isna().sum().to_dict()
    
    grp = out.groupby('City', group_keys=False)
    out[cols] = grp[cols].ffill(limit=3)
    out[cols] = grp[cols].bfill()
    out['AQI'] = grp['AQI'].ffill(limit=3)
    out['AQI'] = grp['AQI'].bfill()
    
    out = out.dropna(subset=['PM2.5', 'AQI']).copy()
    after = out[cols + ['AQI']].isna().sum().to_dict()
    return out, before, after

train_filled, train_null_before, train_null_after = fill_missing_per_split(train_raw, POLLUTANT_COLUMNS)
test_filled, test_null_before, test_null_after = fill_missing_per_split(test_raw, POLLUTANT_COLUMNS)

print('Null counts (train) before:', train_null_before)
print('Null counts (train) after :', train_null_after)
print('Null counts (test) before :', test_null_before)
print('Null counts (test) after  :', test_null_after)
print(f'Rows after null drop -> train: {len(train_filled):,}, test: {len(test_filled):,}')


## Outlier Capping (Bounds Fit On Train Only)

For each city and pollutant, upper cap is `Q3 + 3*IQR` from training data only, then applied to both train and test.

In [ ]:
bounds = {}
for city, grp in train_filled.groupby('City'):
    bounds[city] = {}
    for col in POLLUTANT_COLUMNS:
        series = grp[col].dropna()
        if series.empty:
            bounds[city][col] = np.inf
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        bounds[city][col] = q3 + (3.0 * iqr)

def apply_caps(split_df, caps):
    out = split_df.copy()
    cap_counts = {col: 0 for col in POLLUTANT_COLUMNS}
    
    for city, idx in out.groupby('City').groups.items():
        city_caps = caps.get(city, {})
        for col in POLLUTANT_COLUMNS:
            upper = city_caps.get(col, np.inf)
            if np.isfinite(upper):
                original = out.loc[idx, col]
                capped = original.clip(upper=upper)
                cap_counts[col] += int((original > upper).sum())
                out.loc[idx, col] = capped
    return out, cap_counts

train_capped, train_cap_counts = apply_caps(train_filled, bounds)
test_capped, test_cap_counts = apply_caps(test_filled, bounds)

print('Capped counts by column (train):', train_cap_counts)
print('Capped counts by column (test): ', test_cap_counts)
print('Total capped counts by column:  ', {c: train_cap_counts[c] + test_cap_counts[c] for c in POLLUTANT_COLUMNS})


## Feature Engineering (Safe Temporal Construction)

Lag and rolling features are generated on the chronologically sorted city series using only past values (`shift` and left-aligned rolling over observed history).

In [ ]:
combined_proc = pd.concat([train_capped, test_capped], ignore_index=True)
combined_proc = combined_proc.sort_values(['City', 'Timestamp']).reset_index(drop=True)

grp = combined_proc.groupby('City', group_keys=False)

combined_proc['PM2.5_lag1'] = grp['PM2.5'].shift(1)
combined_proc['PM2.5_lag7'] = grp['PM2.5'].shift(7)
combined_proc['PM2.5_lag30'] = grp['PM2.5'].shift(30)
combined_proc['AQI_lag1'] = grp['AQI'].shift(1)
combined_proc['AQI_lag7'] = grp['AQI'].shift(7)

combined_proc['PM2.5_roll7_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(window=7, min_periods=7).mean())
combined_proc['PM2.5_roll7_std'] = grp['PM2.5'].transform(lambda s: s.rolling(window=7, min_periods=7).std())
combined_proc['PM2.5_roll30_mean'] = grp['PM2.5'].transform(lambda s: s.rolling(window=30, min_periods=30).mean())

combined_proc['day_of_week'] = combined_proc['Timestamp'].dt.dayofweek
combined_proc['month'] = combined_proc['Timestamp'].dt.month
combined_proc['year'] = combined_proc['Timestamp'].dt.year
combined_proc['is_weekend'] = (combined_proc['day_of_week'] >= 5).astype(int)

def month_to_season(m):
    if m in (12, 1, 2):
        return 0
    if m in (3, 4, 5):
        return 1
    if m in (6, 7, 8):
        return 2
    return 3

combined_proc['season'] = combined_proc['month'].apply(month_to_season).astype(int)

before_drop = len(combined_proc)
combined_proc = combined_proc.dropna(subset=LAG_ROLL_COLS).copy()
after_drop = len(combined_proc)
print(f'Dropped rows with lag/rolling nulls: {before_drop - after_drop:,}')
print(f'Rows remaining after feature engineering: {after_drop:,}')


## Re-Split After Feature Generation And Assert Date Integrity

In [ ]:
train_df = combined_proc[(combined_proc['Timestamp'] >= TRAIN_START) & (combined_proc['Timestamp'] <= TRAIN_END)].copy()
test_df = combined_proc[(combined_proc['Timestamp'] >= TEST_START) & (combined_proc['Timestamp'] <= TEST_END)].copy()

assert not train_df.empty, 'Training dataset empty after feature engineering.'
assert not test_df.empty, 'Test dataset empty after feature engineering.'
assert train_df['Timestamp'].max() < test_df['Timestamp'].min(), 'Train/Test overlap after feature generation.'

print(f'Train rows (final, pre-scale): {len(train_df):,}')
print(f'Test rows  (final, pre-scale): {len(test_df):,}')
print('Train date range:', train_df['Timestamp'].min().date(), 'to', train_df['Timestamp'].max().date())
print('Test date range :', test_df['Timestamp'].min().date(), 'to', test_df['Timestamp'].max().date())


## Scale Features (Fit On Train Only) And Save Artifacts

Lag features capture temporal memory and are crucial in autoregressive forecasting. Standardization is fit on training data only to avoid leaking future distribution into model fitting.

In [ ]:
missing_feature_cols = [c for c in FEATURE_COLUMNS if c not in train_df.columns]
if missing_feature_cols:
    raise ValueError(f'Feature columns missing after engineering: {missing_feature_cols}')

scaler = StandardScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_df[FEATURE_COLUMNS].astype(float)),
    columns=FEATURE_COLUMNS,
    index=train_df.index
)
test_scaled = pd.DataFrame(
    scaler.transform(test_df[FEATURE_COLUMNS].astype(float)),
    columns=FEATURE_COLUMNS,
    index=test_df.index
)

for col in FEATURE_COLUMNS:
    train_df[col] = train_scaled[col]
    test_df[col] = test_scaled[col]

joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
with open(MODELS_DIR / 'feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print(f'Saved scaler: {MODELS_DIR / "scaler.pkl"}')
print(f'Saved feature columns: {MODELS_DIR / "feature_columns.json"}')


## Export Train/Test Splits

In [ ]:
export_columns = ['Timestamp', 'City', 'Location', 'AQI'] + FEATURE_COLUMNS

train_out = train_df[export_columns].copy()
test_out = test_df[export_columns].copy()

train_out.to_csv(SPLITS_DIR / 'train.csv', index=False)
test_out.to_csv(SPLITS_DIR / 'test.csv', index=False)

print(f'Saved train split: {SPLITS_DIR / "train.csv"} | rows={len(train_out):,}')
print(f'Saved test split:  {SPLITS_DIR / "test.csv"} | rows={len(test_out):,}')
